# 🌟 Project KISEKI - Training Notebook

**Revolutionary Anime Video Generation on Consumer GPUs**

This notebook provides a complete training pipeline for the KISEKI architecture.

## Architecture Overview

| Component | Innovation |
|:----------|:-----------|
| **Data Loading** | Zero-Copy NVMe Streaming |
| **Backbone** | Mamba-2 SSM (O(N) complexity) |
| **Representation** | SVG-Latent (4000x compression) |
| **Training** | Flow Matching + Self-Improvement |
| **Physics** | Built-in Anime Priors |

## 1. Setup & Installation

In [ ]:
# Install dependencies (run once)
# !pip install torch torchvision torchaudio
# !pip install einops safetensors tqdm rich wandb
# !pip install mamba-ssm  # Requires CUDA
# !pip install flash-attn  # Optional, for faster attention

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 2. Import KISEKI Modules

In [ ]:
from kiseki.core import MambaFlowGenerator
from kiseki.core.mamba_flow import MambaFlowConfig, create_kiseki_600m
from kiseki.tokenizer import AnimeVAE, AnimeVAEConfig, SVGLatentSpace
from kiseki.streaming import ZeroCopyLoader, StreamConfig, IVQIndex
from kiseki.training import (
    FlowMatchingLoss,
    FlowMatchingSampler,
    SelfImproveLoop,
    AnimePriors,
)
from kiseki.utils import MemoryTracker

print('KISEKI modules loaded successfully!')

## 3. Create Model

In [ ]:
# Configuration for development (smaller model)
dev_config = MambaFlowConfig(
    d_model=256,
    d_state=32,
    n_layers=8,
    n_heads=8,
    latent_dim=256,
    expand_factor=2,
    dropout=0.1,
    gradient_checkpointing=True,
)

# Create model
model = MambaFlowGenerator(dev_config)
model = model.to(device)

# Count parameters
n_params = model.get_num_params()
print(f'Model parameters: {n_params:,}')
print(f'Model size: ~{n_params * 2 / 1024**2:.1f} MB (fp16)')

In [ ]:
# Memory tracking
tracker = MemoryTracker()
print(tracker.summary())

## 4. Test Forward Pass

In [ ]:
# Test forward pass
batch_size = 4
seq_len = 32
latent_dim = dev_config.latent_dim

# Random input
x = torch.randn(batch_size, seq_len, latent_dim, device=device)
t = torch.rand(batch_size, device=device)

# Forward
with torch.no_grad():
    output = model(x, t)

print(f'Input shape: {x.shape}')
print(f'Output velocity shape: {output["velocity"].shape}')
print(tracker.summary())

## 5. Setup Training Components

In [ ]:
# Loss function
loss_fn = FlowMatchingLoss()

# Physics priors
physics = AnimePriors(latent_dim=latent_dim).to(device)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01,
    betas=(0.9, 0.95),
)

print('Training components ready!')

## 6. Training Loop (Demo)

In [ ]:
# Demo training loop (with synthetic data)
n_steps = 100
log_interval = 10

model.train()
losses = []

pbar = tqdm(range(n_steps), desc='Training')
for step in pbar:
    # Generate synthetic batch (replace with real data loader)
    x = torch.randn(batch_size, seq_len, latent_dim, device=device)
    
    # Compute loss
    fm_loss = loss_fn(model, x)
    physics_loss = physics.get_loss(x)
    total_loss = fm_loss + 0.1 * physics_loss
    
    # Backward
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    losses.append(total_loss.item())
    
    if (step + 1) % log_interval == 0:
        avg_loss = np.mean(losses[-log_interval:])
        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})

print(f'\nFinal loss: {losses[-1]:.4f}')

In [ ]:
# Plot loss curve
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('KISEKI Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 7. Generation Test

In [ ]:
# Create sampler
sampler = FlowMatchingSampler(
    model,
    n_steps=20,
    cfg_scale=1.0,  # No CFG for unconditional
)

# Generate
model.eval()
with torch.no_grad():
    generated = sampler.sample(
        shape=(1, 16, latent_dim),
        device=device,
    )

print(f'Generated shape: {generated.shape}')
print(f'Generated stats: mean={generated.mean():.4f}, std={generated.std():.4f}')

## 8. VAE Test

In [ ]:
# Create VAE
vae_config = AnimeVAEConfig(
    img_size=256,
    latent_dim=latent_dim,
    hidden_dims=(32, 64, 128, 256),
)

vae = AnimeVAE(vae_config).to(device)

# Test encoding
test_img = torch.randn(1, 3, 256, 256, device=device)
encoded = vae.encode(test_img)

print(f'Original image: {test_img.shape}')
print(f'Encoded z: {encoded["z"].shape}')
print(f'Lineart: {encoded["lineart"].shape}')

# Decode
decoded = vae.decode(encoded['z'])
print(f'Decoded: {decoded.shape}')

## 9. SVG Latent Space Test

In [ ]:
# Create SVG Latent Space
svg_space = SVGLatentSpace(
    latent_dim=latent_dim,
    n_curves=64,
    n_fills=32,
    img_size=256,
).to(device)

# Test
z = torch.randn(1, latent_dim, device=device)
output = svg_space(z)

print(f'Input z: {z.shape}')
print(f'Rendered: {output["rendered"].shape}')
print(f'Compression ratio (1080p): {svg_space.get_compression_ratio(1080):.0f}x')

## 10. Memory Analysis

In [ ]:
from kiseki.utils.memory_utils import estimate_model_memory

print('=== Memory Estimates ===')
print('\nMamba Flow Generator:')
for k, v in estimate_model_memory(model).items():
    print(f'  {k}: {v:.2f} MB')

print('\nAnime VAE:')
for k, v in estimate_model_memory(vae).items():
    print(f'  {k}: {v:.2f} MB')

print('\nSVG Latent Space:')
for k, v in estimate_model_memory(svg_space).items():
    print(f'  {k}: {v:.2f} MB')

print(f'\nCurrent GPU memory: {tracker.summary()}')

## 11. Production Model (600M Parameters)

In [ ]:
# Create production model (use with caution on limited VRAM)
# Uncomment to test:

# tracker.clear_cache()
# prod_model = create_kiseki_600m()
# print(f'Production model parameters: {prod_model.get_num_params():,}')
# print(f'Estimated training memory: {estimate_model_memory(prod_model)["total_training_mb"]:.0f} MB')

## 12. Save Model

In [ ]:
# Save checkpoint
checkpoint_dir = Path('../checkpoints')
checkpoint_dir.mkdir(exist_ok=True)

checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': {
        'model': {
            'd_model': dev_config.d_model,
            'd_state': dev_config.d_state,
            'n_layers': dev_config.n_layers,
            'n_heads': dev_config.n_heads,
            'latent_dim': dev_config.latent_dim,
        }
    }
}

torch.save(checkpoint, checkpoint_dir / 'kiseki_demo.pt')
print(f'Saved checkpoint to {checkpoint_dir / "kiseki_demo.pt"}')

---

## Next Steps

1. **Preprocess your dataset**: `python scripts/preprocess_dataset.py --input_dir /path/to/videos --output_dir ./data/ivq_index`

2. **Full training**: `python scripts/train.py --config configs/model_600m.yaml`

3. **Generate anime**: `python scripts/generate.py --checkpoint ./checkpoints/best.pt --prompt "Your prompt here"`

---

🌸 **Project KISEKI** - *奇跡* (Miracle)